In [ ]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes sentencepiece

In [ ]:
from google.colab import files
uploaded = files.upload()  # Chọn file qa_pairs.jsonl từ máy tính
print('✓ Upload xong:', list(uploaded.keys()))

In [ ]:
from datasets import load_dataset
import json

dataset = load_dataset('json', data_files='qa_pairs.jsonl', split='train')
print(f'Tổng số cặp Q&A: {len(dataset)}')
print('Ví dụ:')
print('  Câu hỏi:', dataset[0]['instruction'])
print('  Trả lời:', dataset[0]['output'][:200])

In [ ]:
def format_example(example):
    text = (
        f"### Câu hỏi:\n{example['instruction']}\n\n"
        f"### Trả lời:\n{example['output']}\n"
    )
    return {'text': text}

dataset = dataset.map(format_example)
print('Ví dụ sau format:')
print(dataset[0]['text'][:300])

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = 'SeaLLMs/SeaLLMs-v3-7B-Chat'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Đang load {model_name}...')
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./lora_output',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    lr_scheduler_type='cosine',
    logging_steps=20,
    save_strategy='epoch',
    bf16=True,
    optim='paged_adamw_8bit',
    report_to='none',
    gradient_checkpointing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()


In [ ]:
trainer.model.save_pretrained('./lora_adapter')
tokenizer.save_pretrained('./lora_adapter')
print('✓ Đã lưu adapter tại ./lora_adapter/')

In [ ]:
def ask(question: str, max_new_tokens: int = 300):
    messages = [
        {"role": "system", "content": "Bạn là trợ lý pháp luật chuyên về Bộ Luật Dân Sự Việt Nam 2015. Trả lời bằng tiếng Việt."},
        {"role": "user", "content": question}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return text.strip()



In [ ]:
from peft import PeftModel

# model = model.merge_and_unload()
model.eval()

In [ ]:
print(ask('Bao nhiêu tuổi được thừa kế ?'))

In [ ]:
from google.colab import files
import json
from rouge_score import rouge_scorer
files.upload()

test_set = json.load(open('test_set.json'))
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

r1_scores, rl_scores = [], []
for item in test_set:
    answer = ask(item['question'])
    scores = scorer.score(item['question'], answer)
    r1_scores.append(scores['rouge1'].fmeasure)
    rl_scores.append(scores['rougeL'].fmeasure)
    print(f"Q: {item['question']}")
    print(f"A: {answer[:200]}")
    print(f"ROUGE-1: {scores['rouge1'].fmeasure:.3f}  ROUGE-L: {scores['rougeL'].fmeasure:.3f}")
    print()

print(f"Avg ROUGE-1: {sum(r1_scores)/len(r1_scores):.3f}")
print(f"Avg ROUGE-L: {sum(rl_scores)/len(rl_scores):.3f}")

In [ ]:
import os
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def evaluate_answer(question, answer):
    prompt = f"""Chấm điểm câu trả lời sau theo thang 1-5:

CÂU HỎI: {question}

CÂU TRẢ LỜI: {answer}

Tiêu chí:
1. Đúng nội dung pháp luật (1-5)
2. Trả lời đúng trọng tâm câu hỏi (1-5)
3. Rõ ràng, mạch lạc (1-5)

Chỉ trả về JSON: {{"content": 0, "relevance": 0, "clarity": 0}}"""

    response = client.models.generate_content(model="gemini-3-flash-preview", contents=prompt)
    import json, re
    raw = response.text.strip()
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
    return json.loads(raw.strip())


results = []
for item in test_set[:10]:
    answer = ask(item['question'])
    scores = evaluate_answer(item['question'], answer)
    results.append(scores)
    print(f"Q: {item['question'][:60]}...")
    print(f"   Content={scores['content']}  Relevance={scores['relevance']}  Clarity={scores['clarity']}")

print(f"\nAvg Content  : {sum(r['content'] for r in results)/len(results):.2f}/5")
print(f"Avg Relevance: {sum(r['relevance'] for r in results)/len(results):.2f}/5")
print(f"Avg Clarity  : {sum(r['clarity'] for r in results)/len(results):.2f}/5")


In [ ]:
import shutil
from google.colab import files
shutil.make_archive('lora_adapter', 'zip', '.', 'lora_adapter')
files.download('lora_adapter.zip')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

# Load model + adapter
print("Đang load model...")
model_name = 'SeaLLMs/SeaLLMs-v3-7B-Chat'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained('./lora_adapter')
base = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map='auto')
model = PeftModel.from_pretrained(base, './lora_adapter')
model.eval()
print("✓ Sẵn sàng!\n")

# Hàm trả lời
def ask(question: str, max_new_tokens: int = 300):
    messages = [
        {"role": "system", "content": "Bạn là trợ lý pháp luật chuyên về Bộ Luật Dân Sự Việt Nam 2015. Trả lời bằng tiếng Việt."},
        {"role": "user", "content": question}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return text.strip()

# Chat loop
print("=" * 50)
print("  CHATBOT BỘ LUẬT DÂN SỰ 2015 (Fine-tuned)")
print("  Gõ 'quit' để thoát")
print("=" * 50)

while True:
    question = input("\nCâu hỏi: ").strip()
    if question.lower() == 'quit':
        print("Tạm biệt!")
        break
    if not question:
        continue
    print(f"\nTrả lời: {ask(question)}")
